# Job Portal Chatbot — Model Training

This notebook trains a TF-IDF + Logistic Regression pipeline on the `JobPortalDataset.csv` file.

### Why the old model had 0% accuracy
- Only 34 rows → 20% test split = ~7 test samples
- Most response classes had 0 or 1 sample, so the classifier could never generalise

### Fix applied
- Dataset expanded to 90+ rows with paraphrased questions (same answers)
- `test_size` reduced to 0.1 so more data goes to training
- `C=10` gives LogisticRegression more flexibility on a small corpus
- A cosine-similarity fallback is added inside `app.py` for unseen queries


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report
import pickle

# Load dataset
file_path = 'JobPortalDataset.csv'
dataset = pd.read_csv(file_path)

print(f'Dataset size: {len(dataset)} rows')
print(f'Unique response classes: {dataset["Response"].nunique()}')
print(dataset.head())

In [ ]:
X = dataset['User Input']
y = dataset['Response']

# Smaller test split so more samples are available per class for training
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.1, random_state=42
)

print(f'Training samples: {len(X_train)}')
print(f'Test samples: {len(X_test)}')

In [ ]:
# TF-IDF captures word importance; LogisticRegression works well on small text corpora
pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(
        ngram_range=(1, 2),   # unigrams + bigrams improve matching
        sublinear_tf=True,    # dampens very frequent terms
        min_df=1
    )),
    ('clf', LogisticRegression(
        solver='lbfgs',
        max_iter=1000,
        C=10              # higher C = less regularisation, better for small dataset
    ))
])

pipeline.fit(X_train, y_train)

y_pred = pipeline.predict(X_test)

print(f'Accuracy: {accuracy_score(y_test, y_pred):.2f}')
print()
print(classification_report(y_test, y_pred, zero_division=0))

In [ ]:
# Save the trained pipeline
with open('job_portal_model.pkl', 'wb') as f:
    pickle.dump(pipeline, f)

print('Model saved as job_portal_model.pkl')

In [ ]:
# Quick smoke test
test_queries = [
    "How do I apply for a job?",
    "I forgot my password",
    "Can I upload my CV?",
    "How to contact support?",
]

for q in test_queries:
    pred = pipeline.predict([q])[0]
    print(f'Q: {q}')
    print(f'A: {pred}')
    print()